In [8]:
import json
import pandas as pd

# Load CSV
csv = pd.read_csv("/home/kuenem/Documents/development/lectures/Master Thesis/Cartogram Framework/data/statistics/india/State_wise_SDP_01_08_2023_Rev.csv")

# Remove accidental whitespace
csv["NAME_1"] = csv["NAME_1"].str.strip()

# Load GeoJSON
with open("/home/kuenem/Documents/development/lectures/Master Thesis/Cartogram Framework/data/geojson/india/states_02.geojson", "r", encoding="utf-8") as f:
    geo = json.load(f)

# Get all NAME_1 values from GeoJSON
geo_names = {
    feature["properties"]["NAME_1"].strip()
    for feature in geo["features"]
}

csv_names = set(csv["NAME_1"])

# Compare
missing_in_geo = csv_names - geo_names
missing_in_csv = geo_names - csv_names

print("In CSV but not GeoJSON:")
print(sorted(missing_in_geo))

print("\nIn GeoJSON but not CSV:")
print(sorted(missing_in_csv))

In CSV but not GeoJSON:
[]

In GeoJSON but not CSV:
['DadraandNagarHaveli', 'DamanandDiu', 'Lakshadweep']


In [9]:
csv_names = set(csv["NAME_1"])

geo["features"] = [
    feature
    for feature in geo["features"]
    if feature["properties"]["NAME_1"].strip() in csv_names
]

with open("states_filtered.geojson", "w", encoding="utf-8") as f:
    json.dump(geo, f, ensure_ascii=False, indent=2)

print(f"Remaining features: {len(geo['features'])}")

Remaining features: 38


In [10]:
import json
from shapely.geometry import shape, mapping, MultiPolygon

# Load GeoJSON
with open("states_filtered.geojson", "r", encoding="utf-8") as f:
    geo = json.load(f)

for feature in geo["features"]:
    geom = shape(feature["geometry"])

    # If it's a MultiPolygon, keep only the largest polygon
    if isinstance(geom, MultiPolygon):
        largest = max(geom.geoms, key=lambda p: p.area)
        feature["geometry"] = mapping(largest)

# Save the modified GeoJSON
with open("states_polygons.geojson", "w", encoding="utf-8") as f:
    json.dump(geo, f, ensure_ascii=False, indent=2)

print("Done.")

Done.
